# Régression : prédiction de `totalLitres`

Notebook pour charger `dataset_cleaned.csv`, préparer les données, entraîner et comparer quatre modèles, évaluer leurs performances et sauvegarder le meilleur modèle.

Placez le CSV dans le même dossier que ce notebook.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib joblib

In [5]:
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "totalLitres"
DATA_PATH = Path(r"C:\Users\pc\Documents\IA cours\projet\dataset\dataset_cleaned.csv")
OUTPUT_DIR = Path(r"C:\Users\ASUS\Downloads\projet IA\dataset\regression_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Optionnel : indiquez vos variables ici. None = toutes les colonnes numériques pertinentes.
FEATURES = None
# Exemple : FEATURES = ["temp", "hum", "moistureLevel", "pres", "waterFlow"]

In [7]:
# 1. Chargement du dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Fichier introuvable : {DATA_PATH.resolve()}. Modifiez DATA_PATH si nécessaire.")

df = pd.read_csv(DATA_PATH, sep=None, engine="python")
df.columns = df.columns.astype(str).str.strip()

print("Dimensions :", df.shape)
print("Colonnes :", df.columns.tolist())
display(df.head())

FileNotFoundError: Fichier introuvable : C:\Users\pc\Documents\IA cours\projet\dataset\dataset_cleaned.csv. Modifiez DATA_PATH si nécessaire.

In [ ]:
# 2. Préparation des données
if TARGET not in df.columns:
    raise ValueError(f"La colonne cible '{TARGET}' est absente.")

data = df.copy()
data[TARGET] = pd.to_numeric(data[TARGET], errors="coerce")
data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET])

if FEATURES is None:
    excluded = {TARGET.lower(), "id", "index", "unnamed: 0", "timestamp", "date", "datetime"}
    feature_names = [c for c in data.select_dtypes(include=[np.number]).columns if c.lower() not in excluded]
else:
    missing = [c for c in FEATURES if c not in data.columns]
    if missing:
        raise ValueError(f"Variables absentes : {missing}")
    feature_names = list(FEATURES)

X = data[feature_names].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
feature_names = [c for c in feature_names if not X[c].isna().all()]
X = X[feature_names]
y = data[TARGET].astype(float)

if len(X) < 5 or X.shape[1] == 0:
    raise ValueError("Données insuffisantes ou aucune variable explicative numérique exploitable.")

print("Cible :", TARGET)
print("Variables explicatives :", feature_names)
print("Nombre de lignes :", len(X))
display(X.isna().sum().to_frame("Valeurs manquantes"))

In [ ]:
# 3. Séparation entraînement / test (80 % / 20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Entraînement :", X_train.shape, "| Test :", X_test.shape)

In [ ]:
# 4. Définition des modèles
linear_prep = ColumnTransformer([("num", Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]), feature_names)])

tree_prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), feature_names)])

models = {
    "Linear Regression": Pipeline([("prep", linear_prep), ("model", LinearRegression())]),
    "Decision Tree": Pipeline([("prep", tree_prep), ("model", DecisionTreeRegressor(max_depth=6, min_samples_leaf=2, random_state=42))]),
    "Random Forest": Pipeline([("prep", tree_prep), ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
    "Gradient Boosting": Pipeline([("prep", tree_prep), ("model", GradientBoostingRegressor(random_state=42))])
}
print("Modèles :", list(models))

In [ ]:
# 5. Entraînement et évaluation
results = []
predictions = {}

for name, model in models.items():
    print("Entraînement :", name)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    mse = mean_squared_error(y_test, pred)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
display(results_df.style.format({"MAE":"{:.4f}", "MSE":"{:.4f}", "RMSE":"{:.4f}", "R2":"{:.4f}"}))

## 6. Comprendre les métriques

- **MAE** : erreur absolue moyenne ; plus petite = mieux.
- **MSE** : moyenne des erreurs au carré ; plus petite = mieux.
- **RMSE** : racine du MSE, dans l'unité de `totalLitres` ; plus petite = mieux.
- **R²** : qualité d'ajustement ; plus proche de 1 = généralement mieux. Un R² négatif indique que le modèle fait moins bien qu'une prédiction constante basée sur la moyenne d'entraînement.

Le classement est basé sur le R² du jeu de test. Examinez aussi MAE et RMSE avant de choisir définitivement un modèle.

In [ ]:
# 7. Choisir le meilleur modèle
best_name = results_df.iloc[0]["Model"]
best_model = models[best_name]
best_predictions = predictions[best_name]
print("Meilleur modèle selon R² :", best_name)
display(results_df.iloc[[0]])

In [ ]:
# 8. Comparaison des scores R²
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["R2"])
plt.title("Comparaison des modèles - R²")
plt.xlabel("Modèle")
plt.ylabel("R²")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 9. Valeurs réelles et prédites
plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.7)
low = min(float(y_test.min()), float(np.min(best_predictions)))
high = max(float(y_test.max()), float(np.max(best_predictions)))
plt.plot([low, high], [low, high], "--")
plt.title(f"Réel vs prédit - {best_name}")
plt.xlabel("Valeurs réelles (totalLitres)")
plt.ylabel("Valeurs prédites (totalLitres)")
plt.tight_layout()
plt.show()

In [ ]:
# 10. Distribution des erreurs
residuals = y_test.to_numpy() - best_predictions
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20, edgecolor="black")
plt.title(f"Distribution des erreurs - {best_name}")
plt.xlabel("Erreur : réel - prédit")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

In [ ]:
# 11. Examiner les prédictions
predictions_df = pd.DataFrame({
    "valeur_reelle": y_test,
    "valeur_predite": best_predictions,
    "erreur": residuals
}, index=y_test.index).sort_index()
display(predictions_df.head(20))

In [ ]:
# 12. Sauvegarder le modèle et les résultats
model_path = OUTPUT_DIR / "best_regression_model.joblib"
joblib.dump({
    "model": best_model,
    "target": TARGET,
    "features": feature_names,
    "model_name": best_name
}, model_path)

results_df.to_csv(OUTPUT_DIR / "resultats_regression.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "predictions_meilleur_modele.csv", index_label="index")

summary = {
    "target": TARGET,
    "best_model": best_name,
    "features": feature_names,
    "metrics": {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v)
                for k, v in results_df.iloc[0].to_dict().items()},
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test))
}
with open(OUTPUT_DIR / "resume_regression.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4, ensure_ascii=False)

# Enregistrer les trois graphiques
plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["R2"])
plt.title("Comparaison des modèles - R²")
plt.ylabel("R²")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparaison_modeles_r2.png", dpi=150)
plt.close()

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.7)
plt.plot([low, high], [low, high], "--")
plt.title(f"Réel vs prédit - {best_name}")
plt.xlabel("Réel (totalLitres)")
plt.ylabel("Prédit (totalLitres)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "reel_vs_predit.png", dpi=150)
plt.close()

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=20, edgecolor="black")
plt.title(f"Distribution des erreurs - {best_name}")
plt.xlabel("Erreur")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribution_erreurs.png", dpi=150)
plt.close()

print("Modèle sauvegardé :", model_path.resolve())
print("Tous les résultats sont dans :", OUTPUT_DIR.resolve())

## 13. Faire une nouvelle prédiction

L'exemple ci-dessous utilise la médiane de chaque variable comme valeur de démonstration. Remplacez ces valeurs par de nouvelles mesures réelles pour obtenir une prédiction utile.

In [ ]:
# Exemple : initialiser avec les médianes des variables du dataset
new_values = {col: float(X[col].median()) for col in feature_names}

# Modifiez les valeurs ci-dessous selon votre nouvelle observation.
new_data = pd.DataFrame([new_values], columns=feature_names)
predicted_total = best_model.predict(new_data)[0]
print(f"Total litres prédit : {predicted_total:.3f}")